<a href="https://colab.research.google.com/github/oscarsanpad/Waveform_optimization_and_design/blob/main/QUBO_HUBO_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
qubo.py  --  ISLR waveform optimization -> QUBO  (BPSK / Q = 2 case)

Pipeline (Huitzillin Quantum Computing, QIS 2026 - Thales SAR track):

    1. Build the ISLR objective  sum_{k>=1} |rho(k)|^2  exactly, as a
       polynomial in spin variables s_i in {+1,-1}  (phase 0 / pi).
       For Q = 2 this is real; rho(k) = sum_i s_i s_{i+k}.

    2. The objective is QUARTIC (degree 4), not quadratic  ->  it is a HUBO,
       not a QUBO. This is the central formulation fact.

    3. Map spins to binary  s_i = 1 - 2 x_i,  x_i in {0,1}.

    4. Rosenberg quadratization: replace each remaining degree>=3 monomial
       by introducing an auxiliary binary variable w = x_a x_b (a logical AND)
       plus a penalty  lam*(x_a x_b - 2 x_a w - 2 x_b w + 3 w),  which is 0
       exactly when w = x_a x_b and positive otherwise. lam must exceed the
       objective's range so feasible minima are never beaten by cheating.

    5. Read off the QUBO matrix Q (upper-triangular; diagonal = linear terms,
       since x_i^2 = x_i) and a constant offset.

    6. Verify against brute force: the QUBO's global minimum must equal the
       true ISLR minimum, and its argmin must be a true optimal code.
"""

import sympy as sp
import itertools


def islr_spin(N):
    """Exact ISLR as a multilinear polynomial in spins s_0..s_{N-1}."""
    s = sp.symbols(f"s0:{N}")
    E = sp.expand(sum(sum(s[i] * s[i + k] for i in range(N - k)) ** 2
                      for k in range(1, N)))
    E = _boolean_or_spin_reduce(E, s, square_to=1)   # s_i^2 = 1
    return E, s


def brute_force_islr(N):
    """Ground-truth ISLR for every binary code of length N."""
    def islr(bits):
        sv = [1 - 2 * b for b in bits]
        return sum((sum(sv[i] * sv[i + k] for i in range(N - k))) ** 2
                   for k in range(1, N))
    table = {b: islr(b) for b in itertools.product([0, 1], repeat=N)}
    m = min(table.values())
    return m, [b for b, v in table.items() if v == m], table


def to_binary(E_spin, s, N):
    """Substitute s_i = 1 - 2 x_i and keep the result multilinear in x."""
    x = sp.symbols(f"x0:{N}")
    Eb = E_spin
    for i in range(N):
        Eb = Eb.subs(s[i], 1 - 2 * x[i])
    Eb = _boolean_or_spin_reduce(sp.expand(Eb), x, square_to="self")  # x_i^2 = x_i
    return Eb, x


def rosenberg_quadratize(Eb, x, lam):
    """Reduce a multilinear binary polynomial to degree <= 2 (a QUBO)."""
    all_vars = list(x)
    poly, penalties, aux, pair_to_aux, kc = Eb, 0, [], {}, 0
    while True:
        poly = sp.expand(poly)
        target = None
        for t in poly.as_ordered_terms():
            facs = [v for v in all_vars if t.has(v)]
            if len(facs) >= 3:
                target = facs
                break
        if target is None:
            break
        a, b = target[0], target[1]
        key = frozenset((a, b))
        if key not in pair_to_aux:
            w = sp.Symbol(f"w{kc}"); kc += 1
            pair_to_aux[key] = w; aux.append(w); all_vars.append(w)
            penalties += lam * (a * b - 2 * a * w - 2 * b * w + 3 * w)
        poly = poly.subs(a * b, pair_to_aux[key])
    H = _boolean_or_spin_reduce(sp.expand(poly + penalties), all_vars, square_to="self")
    return H, all_vars, aux, pair_to_aux


def to_Q_matrix(H, all_vars):
    """Extract QUBO matrix (upper-triangular) and constant offset from H."""
    n = len(all_vars)
    Q = sp.zeros(n, n)
    const = H.subs({v: 0 for v in all_vars})
    for mono, coeff in sp.Poly(H - const, *all_vars).terms():
        idx = [i for i, e in enumerate(mono) if e > 0]
        if len(idx) == 1:
            Q[idx[0], idx[0]] += coeff
        elif len(idx) == 2:
            Q[idx[0], idx[1]] += coeff
    return Q, const


def _boolean_or_spin_reduce(expr, vars_, square_to):
    changed = True
    while changed:
        changed = False
        for v in vars_:
            if expr.has(v ** 2):
                expr = sp.expand(expr.subs(v ** 2, 1 if square_to == 1 else v))
                changed = True
    return expr


def build_and_verify(N, lam=None):
    E_spin, s = islr_spin(N)
    m, argmins, _ = brute_force_islr(N)
    if lam is None:
        lam = 4 * N * N            # comfortably exceeds the ISLR range
    Eb, x = to_binary(E_spin, s, N)
    H, all_vars, aux, pairs = rosenberg_quadratize(Eb, x, lam)
    Q, const = to_Q_matrix(H, all_vars)

    # verify: global QUBO minimum == true ISLR minimum, argmin is optimal
    best = None
    for a in itertools.product([0, 1], repeat=len(all_vars)):
        val = int(H.subs({all_vars[i]: a[i] for i in range(len(all_vars))}))
        if best is None or val < best[0]:
            best = (val, a)
    xmin = tuple(best[1][:N])

    print(f"N = {N},  Q_phase = 2,  penalty lam = {lam}")
    print("ISLR (spin) =", E_spin)
    print(f"true min ISLR = {m}  ({len(argmins)} optimal codes)--- Se refiere a mejor secuencia de las 2^4 disponibles y donde la peor es la 14 al hacer todo 1")
    print(f"variables: {[str(v) for v in x]} + auxiliaries {[str(w) for w in aux]}")
    print(f"QUBO degree = {sp.Poly(H, *all_vars).total_degree()}  "
          f"(2 = valid QUBO)")
    print(f"QUBO global min = {best[0]}  at x = {xmin}")
    print(f"VERIFIED: matches ISLR min = {best[0] == m}, "
          f"x optimal = {xmin in argmins}")
    return Q, const, all_vars


if __name__ == "__main__":
    for N in (4, 5, 6):
        Q, const, order = build_and_verify(N)
        if N == 5:
            print("variable order:", [str(v) for v in order])
            print("constant offset =", const)
            print("Q = 4")
            sp.pprint(Q)
        print("-" * 60)

N = 4,  Q_phase = 2,  penalty lam = 64
ISLR (spin) = 4*s0*s1*s2*s3 + 2*s0*s2 + 2*s1*s3 + 6
true min ISLR = 2  (8 optimal codes)--- Se refiere a mejor secuencia de las 2^4 disponibles y donde la peor es la 14 al hacer todo 1
variables: ['x0', 'x1', 'x2', 'x3'] + auxiliaries ['w0', 'w1']
QUBO degree = 2  (2 = valid QUBO)
QUBO global min = 2  at x = (0, 0, 0, 1)
VERIFIED: matches ISLR min = True, x optimal = True
------------------------------------------------------------
N = 5,  Q_phase = 2,  penalty lam = 100
ISLR (spin) = 4*s0*s1*s2*s3 + 4*s0*s1*s3*s4 + 2*s0*s2 + 2*s0*s4 + 4*s1*s2*s3*s4 + 2*s1*s3 + 2*s2*s4 + 10
true min ISLR = 2  (4 optimal codes)--- Se refiere a mejor secuencia de las 2^4 disponibles y donde la peor es la 14 al hacer todo 1
variables: ['x0', 'x1', 'x2', 'x3', 'x4'] + auxiliaries ['w0', 'w1', 'w2', 'w3']
QUBO degree = 2  (2 = valid QUBO)
QUBO global min = 2  at x = (0, 0, 0, 1, 0)
VERIFIED: matches ISLR min = True, x optimal = True
variable order: ['x0', 'x1', 'x2', '

More Detailed Breakdow:

In [ ]:
import sympy as sp
import itertools

# ======================================================================
#  PART A -- each step of the process ISLR -> QUBO, printed (N = 4, Q = 2)
# ======================================================================
N = 4
print("="*68)
print(f"  ISLR -> QUBO pipeline,  step-by-step   (N = {N}, Q = 2)")
print("="*68)

# --- Stage 1: ISLR in Spin Variables ---
s = sp.symbols(f"s0:{N}")
def rho(k): return sum(s[i]*s[i+k] for i in range(N-k))
E = sp.expand(sum(rho(k)**2 for k in range(1, N)))
ch = True
while ch:
    ch = False
    for si in s:
        if E.has(si**2):
            E = sp.expand(E.subs(si**2, 1)); ch = True
print("\n[1] ISLR in spin variables  (using s_i^2 = 1):")
for k in range(1, N):
    print(f"      rho({k}) = {rho(k)}")
print(f"      ISLR = sum_k rho(k)^2  =  {E}")
print("      ^ The term s0*s1*s2*s3 is DEGREE 4  ->  HUBO, not QUBO")

# --- Etapa 2: to binary ---
x = sp.symbols(f"x0:{N}")
Eb = E
for i in range(N): Eb = Eb.subs(s[i], 1-2*x[i])
Eb = sp.expand(Eb)
ch = True
while ch:
    ch = False
    for xi in x:
        if Eb.has(xi**2):
            Eb = sp.expand(Eb.subs(xi**2, xi)); ch = True
print("\n[2] Replacing  s_i = 1 - 2 x_i   (y x_i^2 = x_i):")
print(f"      ISLR(x) = {Eb}")
print(f"      degree in x = {sp.Poly(Eb,*x).total_degree()}   (still degree 4)")

# --- Steps 3-5: Quadratization printing each auxiliary variable ---
lam = sp.Symbol("lam")
all_vars = list(x)
poly, penalties, pair_to_aux, kc = Eb, 0, {}, 0
print("\n[3-5] Rosenberg quadratization (each auxiliary variable + penalty):")
while True:
    poly = sp.expand(poly)
    target = None
    for t in poly.as_ordered_terms():
        facs = [v for v in all_vars if t.has(v)]
        if len(facs) >= 3:
            target = (t, facs); break
    if target is None:
        break
    t, facs = target
    a, b = facs[0], facs[1]
    key = frozenset((a, b))
    if key not in pair_to_aux:
        w = sp.Symbol(f"w{kc}"); kc += 1
        pair_to_aux[key] = w; all_vars.append(w)
        pen = lam*(a*b - 2*a*w - 2*b*w + 3*w)
        penalties += pen
        print(f"      higher-degree monomial detected (degree >= 3): {t}")
        print(f"        -> introduced {w} = {a}*{b}")
        print(f"        -> substituted {a}*{b} -> {w}   (reduces degree)")
        print(f"        -> added penalty term: {pen}")
    poly = poly.subs(a*b, pair_to_aux[key])

H = sp.expand(poly + penalties)
for v in all_vars:
    ch = True
    while ch:
        ch = False
        if H.has(v**2):
            H = sp.expand(H.subs(v**2, v)); ch = True
print(f"\n      Total objective H(x,w) = reduced_objective + penalties")
print(f"      degree of H = {sp.Poly(H,*all_vars).total_degree()}   (already 2 = QUBO valid)")

# --- Step 6: Q Matrix (with lam = 64) ---
Hn = H.subs(lam, 64)
n = len(all_vars)
Q = sp.zeros(n, n)
const = Hn.subs({v: 0 for v in all_vars})
for mono, coeff in sp.Poly(Hn - const, *all_vars).terms():
    idx = [i for i, e in enumerate(mono) if e > 0]
    if len(idx) == 1: Q[idx[0], idx[0]] += coeff
    elif len(idx) == 2: Q[idx[0], idx[1]] += coeff
print(f"\n[6] Q Matrix  (con lam = 64),  order {[str(v) for v in all_vars]}, offset = {const}:")
sp.pprint(Q)

# ======================================================================
# PART B -- Rosenberg penalty term truth table validation
# ======================================================================
print("\n" + "="*68)
print("  PENALTY TRUTH TABLE:  P(a,b,w) = a*b - 2*a*w - 2*b*w + 3*w")
print("  (with lamda = 1 to see raw values)")
print("="*68)
a_, b_, w_ = sp.symbols("a b w")
P = a_*b_ - 2*a_*w_ - 2*b_*w_ + 3*w_
print(f"\n  {'a':>2} {'b':>2} {'w':>2} | {'a*b':>4} | {'P':>3} | state")
print("  " + "-"*44)
for a, b, w in itertools.product([0,1], repeat=3):
    val = int(P.subs({a_:a, b_:b, w_:w}))
    ok = (w == a*b)
    estado = "OK  (w = a*b)  -> P = 0" if ok else "VIOLATED (w != a*b) -> P > 0"
    print(f"  {a:>2} {b:>2} {w:>2} | {a*b:>4} | {val:>3} | {estado}")
print("\n  Conclusion: P = 0 exactly on the 4 rows where w = a*b,")
print("  and P >= 1 on the 4 rows where it is violated. The minimum penalty for")
print("  violating is 1; multiplied by lam, any invalid state costs >= lambda.")

  ISLR -> QUBO pipeline,  step-by-step   (N = 4, Q = 2)

[1] ISLR in spin variables  (using s_i^2 = 1):
      rho(1) = s0*s1 + s1*s2 + s2*s3
      rho(2) = s0*s2 + s1*s3
      rho(3) = s0*s3
      ISLR = sum_k rho(k)^2  =  4*s0*s1*s2*s3 + 2*s0*s2 + 2*s1*s3 + 6
      ^ The term s0*s1*s2*s3 is DEGREE 4  ->  HUBO, not QUBO

[2] Replacing  s_i = 1 - 2 x_i   (y x_i^2 = x_i):
      ISLR(x) = 64*x0*x1*x2*x3 - 32*x0*x1*x2 - 32*x0*x1*x3 + 16*x0*x1 - 32*x0*x2*x3 + 24*x0*x2 + 16*x0*x3 - 12*x0 - 32*x1*x2*x3 + 16*x1*x2 + 24*x1*x3 - 12*x1 + 16*x2*x3 - 12*x2 - 12*x3 + 14
      degree in x = 4   (still degree 4)

[3-5] Rosenberg quadratization (each auxiliary variable + penalty):
      higher-degree monomial detected (degree >= 3): 64*x0*x1*x2*x3
        -> introduced w0 = x0*x1
        -> substituted x0*x1 -> w0   (reduces degree)
        -> added penalty term: lam*(-2*w0*x0 - 2*w0*x1 + 3*w0 + x0*x1)
      higher-degree monomial detected (degree >= 3): 64*w0*x2*x3
        -> introduced w1 = x2*x3
   

CÓDIGO GENÉTICO

In [ ]:
"""
baseline_ga.py  --  Python translation of Thales' Baseline.m
(Genetic Algorithm to minimize ISLR). Default Q = 2.

1:1 mapping with MATLAB:
  - build_delay_mats  <->  Toeplitz loop (the Q_k matrices)
  - islr              <->  islr(iphi, Q, DelayMats) subroutine
  - genetic_algorithm <->  ga(...) call (population, selection,
                           crossover, mutation, elitism)
"""

import numpy as np
import itertools


# Block 1: Delay matrices Q_k (the 'toeplitz' loop)

def build_delay_mats(N):
    """R[k-1] = NxN matrix with 1s on the k-th superdiagonal (lag k)."""
    R = []
    for k in range(1, N):                       # Lags 1..N-1 (lag 0 is excluded)
        M = np.zeros((N, N))
        for i in range(N - k):
            M[i, i + k] = 1.0                   # Selects the pair (i, i+k)
        R.append(M)
    return R


# Block 2: ISLR subroutine (objective function)

def islr(iphi, Q, R):
    """iphi = phase index vector in {0,...,Q-1}. Returns sum_k |rho(k)|^2."""
    code = np.exp(-1j * 2 * np.pi * np.asarray(iphi) / Q)   # Chips represented as phasors
    ssq = 0.0
    for Mk in R:                                 # Iterate over lags k=1..N-1 / recorre los lags
        rho_k = code @ Mk @ code.conj()          # rho(k) = x Q_k x*  (complex scalar)
        ssq += abs(rho_k) ** 2                    # Accumulates |rho(k)|^2
    return ssq.real


# Block 3: Genetic Algorithm (implements 'ga'/ lo que hace 'ga')
def genetic_algorithm(N, Q, R, pop_size=60, generations=120,
                      pm=0.15, elite=2, tourn=3, seed=0):
    rng = np.random.default_rng(seed)
    # Step 1: Initial random population (genes = phase indices in [0, Q-1])
    pop = rng.integers(0, Q, size=(pop_size, N))
    fit = np.array([islr(ind, Q, R) for ind in pop])   # Step 2: Fitness evaluation (ISLR)

    def tournament():
        cand = rng.integers(0, pop_size, size=tourn)
        return pop[cand[np.argmin(fit[cand])]]         # Candidate with lowest ISLR wins

    best_ind, best_fit = pop[fit.argmin()].copy(), fit.min()
    for _ in range(generations):
        # Elitism: top performers pass through unchanged
        order = np.argsort(fit)
        new = [pop[order[i]].copy() for i in range(elite)]
        while len(new) < pop_size:
            p1, p2 = tournament(), tournament()        # Step 3: selection
            cut = rng.integers(1, N)                    # Step 4: Single-point crossover
            child = np.concatenate([p1[:cut], p2[cut:]])
            for g in range(N):                          # Step 5: mutation
                if rng.random() < pm:
                    child[g] = rng.integers(0, Q)
            new.append(child)
        pop = np.array(new)
        fit = np.array([islr(ind, Q, R) for ind in pop])
        if fit.min() < best_fit:                        # Store global best solution
            best_fit, best_ind = fit.min(), pop[fit.argmin()].copy()
    return best_ind, best_fit


def brute_force(N, Q, R):
    best = None
    for iphi in itertools.product(range(Q), repeat=N):
        v = islr(iphi, Q, R)
        if best is None or v < best[0]:
            best = (v, iphi)
    return best


if __name__ == "__main__":
    N, Q = 15, 2
    R = build_delay_mats(N)

    gi, gf = genetic_algorithm(N, Q, R)
    bf, bi = brute_force(N, Q, R)

    def phases(v): return "-".join("0" if p == 0 else "π" for p in v)
    print(f"N = {N}, Q = {Q}")
    print(f"  Genetic GA  : ISLR = {gf:.0f}  code = {list(gi)}  phases = {phases(gi)}")
    print(f"  True optimum: ISLR = {bf:.0f}  code = {list(bi)}  phases = {phases(bi)}")
    print(f"  Expected Barker-5 (0-0-0-π-0): {'REACHED' if gf == bf else 'no'}")

N = 15, Q = 2
  Genetic GA  : ISLR = 15  code = [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1)]  phases = 0-π-0-0-π-0-0-0-π-0-0-0-π-π-π
  True optimum: ISLR = 15  code = [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1]  phases = 0-0-0-0-0-π-π-0-0-π-π-0-π-0-π
  Expected Barker-5 (0-0-0-π-0): REACHED


In [ ]:
"""
baseline_ga.py -- Classical ISLR baseline (Genetic Algorithm).

Translation of Thales' Baseline.m, instrumented with the same metrics
used for QAOA to enable a direct comparison:
  - Best ISLR found and its ratio against the reference
  - Success rate across multiple seeds (it is a heuristic, like QAOA)
  - Median execution time and number of objective function evaluations
"""

"""
baseline_ga.py -- baseline clasico (algoritmo genetico) del ISLR.

Traduccion del Baseline.m de Thales, instrumentada con las mismas metricas
que usamos para QAOA, para que la comparacion sea directa:
  - mejor ISLR encontrado y su razon contra la referencia
  - tasa de exito sobre varias semillas (es una heuristica, como QAOA)
  - tiempo mediano y numero de evaluaciones de la funcion objetivo
"""

import itertools
import statistics
import time
import numpy as np


# Objective function (identical to the Pyomo model)

def islr(iphi, Q=2):
    """iphi: indices de fase en {0..Q-1}. Devuelve sum_k |rho(k)|^2."""
    code = np.exp(-1j * 2 * np.pi * np.asarray(iphi) / Q)
    N = len(code)
    return float(sum(abs(np.vdot(code[:N - k], code[k:])) ** 2
                     for k in range(1, N)))


def optimo_exacto(N, Q=2):
    """True minimum via enumeration. Only N <= ~18."""
    vals = [islr(c, Q) for c in itertools.product(range(Q), repeat=N)]
    m = min(vals)
    return m, vals.count(m), len(vals)


# The genetic algorithm
def genetico(N, Q=2, pop=60, generaciones=120, pm=0.15, elite=2, torneo=3,
             seed=0):
    rng = np.random.default_rng(seed)
    evals = 0

    pobl = rng.integers(0, Q, size=(pop, N))
    fit = np.array([islr(ind, Q) for ind in pobl]); evals += pop

    def seleccion():
        c = rng.integers(0, pop, size=torneo)
        return pobl[c[np.argmin(fit[c])]]

    mejor_ind, mejor_fit = pobl[fit.argmin()].copy(), fit.min()
    for _ in range(generaciones):
        orden = np.argsort(fit)
        nueva = [pobl[orden[i]].copy() for i in range(elite)]
        while len(nueva) < pop:
            p1, p2 = seleccion(), seleccion()
            corte = rng.integers(1, N)
            hijo = np.concatenate([p1[:corte], p2[corte:]])
            for g in range(N):
                if rng.random() < pm:
                    hijo[g] = rng.integers(0, Q)
            nueva.append(hijo)
        pobl = np.array(nueva)
        fit = np.array([islr(ind, Q) for ind in pobl]); evals += pop
        if fit.min() < mejor_fit:
            mejor_fit, mejor_ind = fit.min(), pobl[fit.argmin()].copy()
    return mejor_ind, mejor_fit, evals


# Multi-start runs
def evaluar(N, semillas=(1, 2, 3, 4, 5), Q=2, **kw):
    opt, n_opt, espacio = optimo_exacto(N, Q)
    ref = (N - 1) / 2                         # Reference Barker value from specifications

    fits, tiempos, evals_l, codigos = [], [], [], []
    for s in semillas:
        t0 = time.time()
        ind, f, ev = genetico(N, Q, seed=s, **kw)
        tiempos.append(time.time() - t0)
        fits.append(f); evals_l.append(ev); codigos.append(list(ind))

    exitos = sum(1 for f in fits if abs(f - opt) < 1e-9)
    mejor = min(fits)

    print(f"N = {N}   ({espacio} secuences, {len(semillas)} runs)")
    print(f"  True optimal ISLR .................. {opt:g}"
          f"   ({n_opt} sequences reach it)")
    print(f"  Barker reference (N-1)/2 ......... {ref:g}"
          f"   {'(achievable)' if abs(opt-ref) < 1e-9 else '(NO Barker code exists)'}")
    print(f"  best ISLR found ................ {mejor:g}"
          f"   -> ratio vs optimal = {mejor/opt:.2f}")
    print(f"  median / worst ISLR ............... {statistics.median(fits):g}"
          f" / {max(fits):g}")
    print(f"  success rate ..................... {exitos}/{len(semillas)}")
    print(f"  median execution time .................... {statistics.median(tiempos):.2f} s")
    print(f"  objective function evaluations ......... {evals_l[0]}"
          f"   ({evals_l[0]/espacio:.1%} of search space)")
    print(f"  best code/sequence ...................... "
          f"{codigos[int(np.argmin(fits))]}")
    print()
    return {"N": N, "optimal": opt, "best": mejor,
            "median": statistics.median(fits),
            "successes": exitos, "runs": len(semillas),
            "t_median": statistics.median(tiempos), "evals": evals_l[0]}


if __name__ == "__main__":
    print("=" * 62)
    print("  GENETIC BASELINE -- QAOA-comparable metrics")
    print("=" * 62)
    print()
    filas = [evaluar(N) for N in (5, 7, 11, 15)]
    print("Summary:")
    print(f"{'N':>3} {'optimal':>7} {'best':>7} {'median':>8} {'success':>7} {'t(s)':>7}")
    for r in filas:
        print(f"{r['N']:>3} {r['optimal']:>7g} {r['best']:>7g} "
              f"{r['median']:>8g} {r['successes']}/{r['runs']:<5} "
              f"{r['t_median']:>7.2f}")

  GENETIC BASELINE -- QAOA-comparable metrics

N = 5   (32 secuences, 5 runs)
  True optimal ISLR .................. 2   (4 sequences reach it)
  Barker reference (N-1)/2 ......... 2   (achievable)
  best ISLR found ................ 2   -> ratio vs optimal = 1.00
  median / worst ISLR ............... 2 / 2
  success rate ..................... 5/5
  median execution time .................... 0.40 s
  objective function evaluations ......... 7260   (22687.5% of search space)
  best code/sequence ...................... [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0)]

N = 7   (128 secuences, 5 runs)
  True optimal ISLR .................. 3   (4 sequences reach it)
  Barker reference (N-1)/2 ......... 3   (achievable)
  best ISLR found ................ 3   -> ratio vs optimal = 1.00
  median / worst ISLR ............... 3 / 3
  success rate ..................... 5/5
  median execution time .................... 0.43 s
  objective function evaluations ......... 7260   (5671.